<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/LangChain(Retrieveres%26Loaders).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q -U langchain langchain-chroma langchain-community langchain-text-splitters wikipedia arxiv

In [2]:
!pip install -q -U \
    opentelemetry-api \
    opentelemetry-sdk \
    opentelemetry-semantic-conventions \
    opentelemetry-exporter-otlp-proto-grpc

In [3]:
import arxiv

from langchain_community.retrievers import WikipediaRetriever
from langchain_core.documents import Document

/tmp/ipykernel_21637/102684421.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.retrievers import WikipediaRetriever


In [4]:
def search_arxiv(query: str, top_k_results: int = 5):
    client = arxiv.Client()

    search = arxiv.Search( # описваме заявката, като поръчка е Search
        query = query,
        max_results= top_k_results,
        sort_by= arxiv.SortCriterion.Relevance # подрежда най-ревелантните
    )
# arxiv.SortCriterion.SubmittedDate: подреди по дата на публикуване
    documents = []
# client.results(search) подаваме на клиента тази search заявка - да търси
    for result in client.results(search):
        documents.append(
            Document(
                page_content = result.summary,
                metadata = {
                    "title": result.title,
                    "authors": [author.name for author in result.authors],
                    "published": str(result.published),
                    "source": result.entry_id
                }
            )
        )

    return documents

In [5]:
# Retrievers търсене на инф повреме на inference(когато ai modela подготвя отговор за нас)

wikipedia_retriever = WikipediaRetriever(top_k_results = 20, doc_content_chars_max=0)

In [6]:
arxiv_docs = search_arxiv("transformers", top_k_results=5)

arxiv_docs

[Document(metadata={'title': 'Physics-Informed Machine Learning for Transformer Condition Monitoring -- Part I: Basic Concepts, Neural Networks, and Variants', 'authors': ['Jose I. Aizpurua'], 'published': '2025-12-20 10:10:25+00:00', 'source': 'http://arxiv.org/abs/2512.22190v1'}, page_content='Power transformers are critical assets in power networks, whose reliability directly impacts grid resilience and stability. Traditional condition monitoring approaches, often rule-based or purely physics-based, struggle with uncertainty, limited data availability, and the complexity of modern operating conditions. Recent advances in machine learning (ML) provide powerful tools to complement and extend these methods, enabling more accurate diagnostics, prognostics, and control. In this two-part series, we examine the role of Neural Networks (NNs) and their extensions in transformer condition monitoring and health management tasks. This first paper introduces the basic concepts of NNs, explores C

In [7]:
wikipedia_retriever.invoke(input="transformers")

/usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py:389: GuessedAtParserWarning: No parser was explicitly specified, so I'm using the best available HTML parser for this system ("lxml"). This usually isn't a problem, but if you run this code on another system, or in a different virtual environment, it may use a different parser and behave differently.

The code that caused this warning is on line 389 of the file /usr/local/lib/python3.12/dist-packages/wikipedia/wikipedia.py. To get rid of this warning, pass the additional argument 'features="lxml"' to the BeautifulSoup constructor.

  lis = BeautifulSoup(html).find_all('li')


[Document(metadata={'title': 'Transformers', 'summary': 'Transformers is a mecha media franchise produced by American toy company Hasbro and Japanese toy company Takara Tomy. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue, making it one of the highest-grossing media franchises of all time.\nThe franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara\'s Diaclone and Micro Change toylines rebranded for Western markets. The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic b

In [8]:
wikipedia_retriever.invoke(input="transformers")

[Document(metadata={'title': 'Transformers', 'summary': 'Transformers is a mecha media franchise produced by American toy company Hasbro and Japanese toy company Takara Tomy. It primarily follows the heroic Autobots and the villainous Decepticons, two alien robot factions at war that can transform into other forms, such as vehicles and animals. The franchise encompasses toys, animation, comic books, video games and films. As of 2011, it generated more than ¥2 trillion ($25 billion) in revenue, making it one of the highest-grossing media franchises of all time.\nThe franchise began in 1984 with the Transformers toy line, comprising transforming mecha toys from Takara\'s Diaclone and Micro Change toylines rebranded for Western markets. The term "Generation 1" (G1) covers both the animated television series The Transformers and the comic book series of the same name, which are further divided into Japanese, British and Canadian spin-offs. Sequels followed, such as the Generation 2 comic b

In [9]:
from langchain_core.tools import create_retriever_tool

In [10]:
create_retriever_tool(wikipedia_retriever, name = "search_wikipedia", description ="Use this tool to search for related pages in Wikipedia.")

StructuredTool(name='search_wikipedia', description='Use this tool to search for related pages in Wikipedia.', args_schema=<class 'langchain_core.tools.retriever.RetrieverInput'>, func=<function create_retriever_tool.<locals>.func at 0x7bd83ac15bc0>, coroutine=<function create_retriever_tool.<locals>.afunc at 0x7bd83ac15b20>)

## Loaders

In [11]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

In [16]:
# Loaders <-> import (into vector db)
# Loaders -> Split -> Embed -> Import data into Vector DB -> Retriever -> Tool -> Agent

directory_loader = DirectoryLoader("/content/books", glob = "*.txt", recursive = True, loader_cls= TextLoader)
text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 300)
vector_store = Chroma(collection_name = "book_fragments", persist_directory="/content/chroma")

In [17]:
books = directory_loader.load() # зареждаме документите и получавеме списък от документи

In [14]:

text_splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 300)

In [15]:
chunked_books = text_splitter.split_documents(books)

In [18]:
from itertools import batched
# да влизат чънковете от книгите на бачове по 250
for index, batch in enumerate(batched(chunked_books, 250)):
    vector_store.add_documents(batch)
    print(f"Processed batch #{index + 1}")

/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 45.7MiB/s]


Processed batch #1
Processed batch #2
Processed batch #3
Processed batch #4
Processed batch #5
Processed batch #6
Processed batch #7
Processed batch #8
Processed batch #9
Processed batch #10
Processed batch #11
Processed batch #12
Processed batch #13
Processed batch #14
Processed batch #15
Processed batch #16
Processed batch #17
Processed batch #18
Processed batch #19
Processed batch #20
Processed batch #21
Processed batch #22
Processed batch #23
Processed batch #24
Processed batch #25
Processed batch #26
Processed batch #27
Processed batch #28
Processed batch #29
Processed batch #30


In [19]:
vector_store.search("newspaper", search_type = "similarity")

[Document(id='f2fdca6e-74a9-4afc-b389-f296c8117166', metadata={'source': '/content/books/Crime and Punishment.txt'}, page_content='“Ah, you are wrong! I got the clothes before. It was the new clothes in\nfact that made me think of taking you in.”\n\n“Are you such a good dissembler?” Raskolnikov asked carelessly.\n\n“You wouldn’t have supposed it, eh? Wait a bit, I shall take you in,\ntoo. Ha-ha-ha! No, I’ll tell you the truth. All these questions about\ncrime, environment, children, recall to my mind an article of yours\nwhich interested me at the time. ‘On Crime’... or something of the\nsort, I forget the title, I read it with pleasure two months ago in the\n_Periodical Review_.”\n\n“My article? In the _Periodical Review_?” Raskolnikov asked in\nastonishment. “I certainly did write an article upon a book six months\nago when I left the university, but I sent it to the _Weekly Review_.”\n\n“But it came out in the _Periodical_.”\n\n“And the _Weekly Review_ ceased to exist, so that’s why

In [24]:
book_retriever = vector_store.as_retriever(search_kwargs = {"k":10})

In [25]:
book_retriever.invoke("newspaper")

[Document(id='f2fdca6e-74a9-4afc-b389-f296c8117166', metadata={'source': '/content/books/Crime and Punishment.txt'}, page_content='“Ah, you are wrong! I got the clothes before. It was the new clothes in\nfact that made me think of taking you in.”\n\n“Are you such a good dissembler?” Raskolnikov asked carelessly.\n\n“You wouldn’t have supposed it, eh? Wait a bit, I shall take you in,\ntoo. Ha-ha-ha! No, I’ll tell you the truth. All these questions about\ncrime, environment, children, recall to my mind an article of yours\nwhich interested me at the time. ‘On Crime’... or something of the\nsort, I forget the title, I read it with pleasure two months ago in the\n_Periodical Review_.”\n\n“My article? In the _Periodical Review_?” Raskolnikov asked in\nastonishment. “I certainly did write an article upon a book six months\nago when I left the university, but I sent it to the _Weekly Review_.”\n\n“But it came out in the _Periodical_.”\n\n“And the _Weekly Review_ ceased to exist, so that’s why